# Занятие 24. Практика: признаки для цены квартиры (~90 мин)

Вы **пишете весь код сами**. Ячейку **«Дано»** не меняйте.

Задача — регрессия: предсказать **цену** квартиры (млн руб.) по объявлению. Модель для проверки идей — kNN-регрессия (занятие 21), метрика — **MAE**. Теория — `Урок_23_Feature_Engineering_Теория.ipynb` (занятие 23).

Ориентир по времени указан у каждого задания. Если застряли — идите дальше и вернитесь позже.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|---|---:|
| 0 | Постановка | 2 |
| 1 | Split | 3 |
| 2 | Числовые признаки | 3 |
| 3 | Признаки из даты | 3 |
| 4 | Категории | 4 |
| 5 | Пропуски | 3 |
| 6 | Baseline-набор и kNN | 4 |
| 7 | Добавляем группы признаков | 4 |
| 8 | Комбинированный набор | 2 |
| 9 | Итог | 2 |

---
## Дано: датасет

Синтетическая таблица объявлений (300 строк): площадь, комнаты, район, состояние, балкон, дата публикации, просмотры за месяц, доход района (есть пропуски) и **цена** — целевая переменная. Момент прогноза — `PREDICTION_DATE`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

districts = rng.choice(['центр', 'спальный', 'пригород'], size=n, p=[0.3, 0.5, 0.2])
condition = rng.choice(['требует ремонта', 'среднее', 'хорошее'], size=n, p=[0.2, 0.5, 0.3])
area = rng.uniform(25, 130, n).round(0)
rooms = np.clip((area / 30).astype(int) + rng.integers(0, 2, n), 1, 5)
balcony = rng.integers(0, 2, n)
listing_date = pd.Timestamp('2026-01-01') + pd.to_timedelta(rng.integers(0, 180, n), unit='D')
views = np.expm1(rng.normal(4.0, 1.2, n)).round(0)

district_bonus = pd.Series(districts).map({'центр': 4.0, 'спальный': 1.0, 'пригород': 0.0}).values
condition_bonus = pd.Series(condition).map(
    {'требует ремонта': -1.5, 'среднее': 0.0, 'хорошее': 1.5}
).values
price = (0.13 * area + 0.02 * area * balcony + district_bonus
         + condition_bonus + rng.normal(0, 1.0, n)).round(1)

district_income = pd.Series(districts).map({'центр': 95.0, 'спальный': 60.0, 'пригород': 45.0}).values
district_income = district_income + rng.normal(0, 5, n).round(0)
missing_mask = rng.random(n) < 0.15
district_income[missing_mask] = np.nan

flats = pd.DataFrame({
    'площадь': area,
    'комнаты': rooms,
    'район': districts,
    'состояние': condition,
    'балкон': balcony,
    'дата': listing_date,
    'просмотры_за_месяц': views,
    'доход_района': district_income,
    'цена': price,
})

PREDICTION_DATE = pd.Timestamp('2026-07-01')
RANDOM_STATE = 42
print('Объектов:', len(flats))
flats.head()


---
## Задание 0. Постановка (~5 мин) — **2 балла**

Перед тем как строить признаки, зафиксируйте постановку задачи обычным текстом. Напишите в markdown-ячейке, что одной строкой данных является объявление о квартире, какую величину мы хотим предсказывать и почему такая цель делает задачу регрессией. Отдельно укажите, в какой момент времени модель должна делать прогноз: это важно, потому что признаки должны быть доступны именно к этому моменту.

Завершите ответ примером утечки данных. Можно разобрать признак `цена_за_метр = цена / площадь`: если `цена` является целевой переменной, такой признак уже содержит часть правильного ответа и поэтому не может использоваться при обучении модели.

### Подробные критерии (для проверки LLM)

- **0.5 балла** — назван объект наблюдения: одно объявление о квартире.
- **0.5 балла** — указана цель и объяснено, что это регрессия.
- **0.5 балла** — указан момент прогноза: признаки известны к публикации объявления.
- **0.5 балла** — приведён и объяснён пример утечки данных.

### Снижение баллов
- Нет примера утечки или объяснения, где подглядывается ответ → минус **0.5**.
- Тип задачи назван неверно → минус **0.5**.

**Ответ:**

Одна строка — одно объявление о продаже квартиры. Модель должна предсказать цену квартиры в млн рублей, поэтому это задача регрессии: целевая переменная числовая, а не класс. Прогноз делается в момент публикации объявления, когда известны характеристики квартиры и объявления, но неизвестно будущее поведение рынка. Признак `цена_за_метр = цена / площадь` использовать нельзя, потому что он напрямую содержит целевую переменную `цена`: модель фактически получает часть правильного ответа.

**Ответ:**

Признак `цена_за_метр = цена / площадь` использовать нельзя, потому что в нём напрямую участвует целевая переменная `цена`. Такой признак подглядывает в ответ: модель получает часть правильного значения ещё до прогноза и на validation будет выглядеть лучше, чем в реальном использовании.


---
## Задание 1. Split (~7 мин) — **3 балла**

Сделайте первое рабочее разбиение данных: отделите от таблицы `flats` обучающую часть `flats_train` и валидационную часть `flats_val`. Используйте соотношение 70/30 и `random_state=RANDOM_STATE`, чтобы результат был воспроизводимым.

После разбиения выведите размеры обеих таблиц и проверьте, что дальше все обучаемые преобразования будут настраиваться только по `flats_train`. Валидационная часть нужна для честной проверки качества, а не для подглядывания при подготовке признаков.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — создана обучающая таблица flats_train.
- **1.0 балл** — создана валидационная таблица flats_val.
- **1.0 балл** — использованы разбиение 70/30 и RANDOM_STATE; размеры близки к 210 / 90.

### Снижение баллов
- Обучаемые преобразования настраиваются не только по train → минус **1.0**.
- Размеры выборок не выведены → минус **0.5**.

In [ ]:
from sklearn.model_selection import train_test_split

flats_train, flats_val = train_test_split(flats, test_size=0.3, random_state=RANDOM_STATE)
print('train:', len(flats_train), '| validation:', len(flats_val))


---
## Задание 2. Числовые признаки (~10 мин) — **3 балла**

Добавьте в обе таблицы, `flats_train` и `flats_val`, два числовых признака. Первый признак, `площадь_на_комнату`, должен показывать, сколько квадратных метров в среднем приходится на одну комнату. Второй признак, `log_просмотры`, должен быть логарифмом `просмотры_за_месяц` через `np.log1p`, чтобы большие значения просмотров не слишком сильно доминировали.

После создания признаков убедитесь, что новые столбцы появились в обеих таблицах и в них нет пропусков. Эти признаки не требуют обучения на данных, поэтому одну и ту же формулу можно применить к train и validation.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — создан признак площадь_на_комнату в train и validation.
- **1.0 балл** — создан log_просмотры через np.log1p в train и validation.
- **1.0 балл** — проверено, что новые признаки появились и не содержат NaN.

### Снижение баллов
- Признак создан только в одной из таблиц → минус **0.5** за каждый случай.
- Использован обычный log вместо np.log1p → минус **0.5**.

In [ ]:
for table in (flats_train, flats_val):
    table['площадь_на_комнату'] = table['площадь'] / table['комнаты']
    table['log_просмотры'] = np.log1p(table['просмотры_за_месяц'])
flats_train[['площадь_на_комнату', 'log_просмотры']].describe().round(2)


---
## Задание 3. Признаки из даты (~10 мин) — **3 балла**

Преобразуйте дату публикации объявления в простые числовые признаки, которые модель сможет использовать без работы с типом datetime. В обеих таблицах выделите номер месяца в столбец `месяц`: это позволит модели учитывать сезонность объявлений хотя бы в базовом виде.

Также посчитайте `дней_с_публикации`: это разница между `PREDICTION_DATE` и датой публикации объявления. Проверьте, что получившееся число неотрицательное для всех строк, иначе это означало бы ошибку во времени прогноза или в данных.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — выделен `месяц` из даты в train и validation.
- **1.0 балл** — создан `дней_с_публикации` в train и validation.
- **1.0 балл** — проверено, что `дней_с_публикации` неотрицателен.

### Снижение баллов
- Признаки даты построены только для train или только для validation → минус **0.5**.
- `дней_с_публикации` посчитан в обратную сторону → минус **0.5**.

In [ ]:
for table in (flats_train, flats_val):
    table['месяц'] = table['дата'].dt.month
    table['дней_с_публикации'] = (PREDICTION_DATE - table['дата']).dt.days

assert (flats_train['дней_с_публикации'] >= 0).all()
assert (flats_val['дней_с_публикации'] >= 0).all()

flats_train[['месяц', 'дней_с_публикации']].head()

---
## Задание 4. Категории (~12 мин) — **4 балла**

Закодируйте категориальные признаки так, чтобы их можно было передать в модель. Для признака `район` используйте `OneHotEncoder(handle_unknown='ignore')`: обучите кодировщик только на `flats_train`, а затем примените его и к train, и к validation. Такой режим нужен, чтобы новая или редкая категория в validation не ломала код.

Для признака `состояние` используйте `OrdinalEncoder` с явным порядком категорий: `требует ремонта < среднее < хорошее`. Добавьте в обе таблицы столбец `состояние_код` и проверьте, что порядок отражает смысл признака, а не случайный алфавитный порядок.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — OneHotEncoder с handle_unknown='ignore' обучен на flats_train.
- **1.0 балл** — one-hot признаки района добавлены к train и validation.
- **1.0 балл** — OrdinalEncoder использует порядок требует ремонта < среднее < хорошее.
- **1.0 балл** — состояние_код создан в обеих таблицах.

### Снижение баллов
- Кодировщик обучен на validation или всей таблице до split → минус **1.0**.
- Для состояния использован случайный или алфавитный порядок → минус **0.5**.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

district_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
district_train = district_encoder.fit_transform(flats_train[['район']])
district_val = district_encoder.transform(flats_val[['район']])
district_cols = list(district_encoder.get_feature_names_out())

cond_encoder = OrdinalEncoder(categories=[['требует ремонта', 'среднее', 'хорошее']])
flats_train['состояние_код'] = cond_encoder.fit_transform(flats_train[['состояние']])
flats_val['состояние_код'] = cond_encoder.transform(flats_val[['состояние']])

for table, encoded in ((flats_train, district_train), (flats_val, district_val)):
    for j, col in enumerate(district_cols):
        table[col] = encoded[:, j]
print('One-hot столбцы:', district_cols)


---
## Задание 5. Пропуски (~8 мин) — **3 балла**

Обработайте пропуски в признаке `доход_района` так, чтобы модель получила и заполненное значение, и информацию о самом факте пропуска. В обеих таблицах создайте индикатор `доход_пропущен`, где 1 означает, что исходное значение было пропущено, а 0 — что оно было известно.

Затем создайте `доход_заполнен`: замените пропуски медианой, посчитанной только по `flats_train`. Не пересчитывайте медиану на validation, потому что в рабочем сценарии параметры обработки должны приходить из train и затем одинаково применяться к новым данным.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — создан доход_пропущен в train и validation.
- **1.0 балл** — медиана дохода посчитана только по train.
- **1.0 балл** — создан доход_заполнен, пропусков в нём не осталось.

### Снижение баллов
- Медиана посчитана по train+validation или отдельно по validation → минус **1.0**.
- Индикатор пропуска не создан → минус **0.5**.

In [ ]:
median_income = flats_train['доход_района'].median()
for table in (flats_train, flats_val):
    table['доход_пропущен'] = table['доход_района'].isna().astype(int)
    table['доход_заполнен'] = table['доход_района'].fillna(median_income)
assert flats_val['доход_заполнен'].notna().all()
print('Медиана train:', round(median_income, 1))


---
## Задание 6. Baseline-набор и kNN (~10 мин) — **4 балла**

Постройте первый честный baseline для регрессии. В качестве минимального набора признаков возьмите только `площадь` и `комнаты`: это простая точка отсчёта, с которой потом можно сравнивать новые признаки.

Соберите `Pipeline`, где сначала стоит `StandardScaler`, а затем `KNeighborsRegressor(n_neighbors=5)`. Обучите pipeline на train и посчитайте MAE на validation, сохранив результат в `mae_base`. Pipeline здесь важен: масштабирование должно обучаться на train и затем теми же параметрами применяться к validation.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — выбран baseline-набор площадь + комнаты.
- **1.0 балл** — собран Pipeline из StandardScaler и KNeighborsRegressor.
- **1.0 балл** — модель обучена на train.
- **1.0 балл** — mae_base посчитан на validation.

### Снижение баллов
- Масштабирование обучено до split или на validation → минус **1.0**.
- MAE посчитан не на validation → минус **1.0**.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def validation_mae(features):
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=5)),
    ])
    model.fit(flats_train[features], flats_train['цена'])
    return mean_absolute_error(flats_val['цена'], model.predict(flats_val[features]))

base_features = ['площадь', 'комнаты']
mae_base = validation_mae(base_features)
print('MAE базовый набор:', round(mae_base, 3), 'млн')


---
## Задание 7. Добавляем группы признаков (~12 мин) — **4 балла**

Проверьте, какие группы признаков действительно помогают baseline-модели. Для каждой группы соберите набор «базовые признаки плюс одна новая группа», обучите ту же kNN-модель и измерьте MAE на validation.

Сравните три варианта: числовые признаки `площадь_на_комнату` и `log_просмотры`; признаки `состояние_код` и `балкон`; one-hot столбцы районов. Результат оформите таблицей «набор признаков → MAE», чтобы по числам было видно, какие идеи улучшили модель, а какие нет.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — проверена группа числовых признаков.
- **1.0 балл** — проверена группа состояние_код + балкон.
- **1.0 балл** — проверена группа one-hot районов.
- **1.0 балл** — собрана таблица набор → MAE и есть вывод по validation.

### Снижение баллов
- Сравнение сделано не на validation → минус **1.0**.
- Нет таблицы сравнения или вывода по числам → минус **0.5**.

In [ ]:
candidate_groups = {
    'числовые': ['площадь_на_комнату', 'log_просмотры'],
    'состояние+балкон': ['состояние_код', 'балкон'],
    'район (one-hot)': district_cols,
}
results = {'база': mae_base}
for name, group in candidate_groups.items():
    results[name] = validation_mae(base_features + group)
pd.Series(results).round(3).to_frame('MAE, млн')


**Вывод:**

Сравнивать группы нужно по validation MAE: чем меньше MAE, тем полезнее группа признаков. В этих данных сильнее всего должны помогать `состояние+балкон` и one-hot района, потому что они связаны с формулой цены; числовые производные признаки полезны только если уменьшают MAE относительно базового набора.


---
## Задание 8. Комбинированный набор (~8 мин) — **2 балла**

Соберите финальный набор признаков не автоматически «всё подряд», а по результатам предыдущего сравнения. Начните с базовых признаков и добавьте только те группы, которые улучшили MAE в задании 7.

После этого снова обучите модель на train, посчитайте `mae_final` на validation и сравните его с `mae_base`. Важно показать не только число, но и логику выбора: какие группы вошли в финальный набор и почему.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — финальный набор собран из baseline и групп, улучшивших MAE в задании 7.
- **1.0 балл** — mae_final посчитан на validation и сравнен с mae_base.

### Снижение баллов
- В финальный набор добавлены группы без опоры на результаты задания 7 → минус **0.5**.
- Нет сравнения mae_final и mae_base → минус **0.5**.

In [ ]:
final_features = base_features + candidate_groups['состояние+балкон'] + candidate_groups['район (one-hot)']
mae_final = validation_mae(final_features)
print('Финальный набор:', final_features)
print('MAE база:', round(mae_base, 3), '→ MAE финал:', round(mae_final, 3))


**Вывод:**

Финальный набор стоит собирать только из тех групп, которые улучшили MAE на validation. Если `mae_final` меньше `mae_base`, новые признаки действительно помогли модели; если нет, значит часть добавленных признаков добавила шум и её лучше не включать.


---
## Задание 9. Итог (~8 мин) — **2 балла**

Напишите итоговый вывод в markdown-ячейке. Сначала перечислите группы признаков, которые улучшили MAE, и группы, которые не дали улучшения; объясните это как гипотезу, опираясь на таблицу из заданий 7–8.

Затем отдельно назовите признак, который вы не стали строить из-за утечки данных, и объясните, где именно он подглядывает ответ. В конце перечислите параметры обработки, которые нужно сохранить для применения модели к завтрашним объявлениям: обученные кодировщики категорий, медиану train для пропусков и параметры масштабирования внутри pipeline.

### Подробные критерии (для проверки LLM)

- **0.7 балла** — перечислены группы признаков, которые улучшили MAE, и группы без улучшения.
- **0.5 балла** — объяснение опирается на числа из заданий 7–8.
- **0.4 балла** — назван признак, исключённый из-за утечки данных.
- **0.4 балла** — перечислены параметры обработки, которые нужно сохранить для новых объявлений.

### Снижение баллов
- Вывод не связан с полученными MAE → минус **0.5**.
- Не упомянуто, что обучаемые параметры обработки берутся с train → минус **0.5**.

**Эталонные тезисы:**

1. Состояние+балкон и район дают наибольший вклад — они напрямую входят в формулу цены; log-просмотры почти не помогают (просмотры не влияют на цену в этих данных).
2. «Цена за метр» — содержит цель; агрегат «средняя цена района по всем данным» — заглядывает в будущее и в validation.
3. Сохранить и применять: обученные на train кодировщики (`OneHotEncoder`, `OrdinalEncoder`), медиану дохода train, `StandardScaler` внутри pipeline; новые категории районов обрабатываются `handle_unknown='ignore'`.